<a href="https://colab.research.google.com/github/icervecon/Dividend-Design-UCITS-Funds/blob/main/Paper_Dividend_Design_UCITS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dividend Policies as Product Design: Evidence from UCITS Funds and the Role of Asset Management Firms

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
file_path = "ucits_dividend_data.xlsx"
df = pd.read_excel(file_path)

In [ ]:
df['dividend'] = df['has_paid_dividends_i']
df['CL4_yes'] = (df['CL4'] == 'Yes').astype(int)

Table 1- Descriptive Statistics- Panel A- Full Sample

In [ ]:
vars_desc = [
    'dividend',
    'retail_eligible_i',
    'CL4_yes',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income'
]

desc = df[vars_desc].describe().T

desc = desc[['mean', 'std', 'min', 'max']]
desc

Table 1- Descriptive Statistics- Panel B- By Retail Eligibility

In [ ]:
vars_desc = [
    'dividend',
    'CL4_yes',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income'
]

grouped = df.groupby('retail_eligible_i')[vars_desc].mean().T

grouped.columns = ['Non-Retail', 'Retail']

grouped['Difference'] = grouped['Retail'] - grouped['Non-Retail']

grouped

Table 2. Accessibility and Dividend Policies (Baseline)

In [ ]:
import statsmodels.formula.api as smf

# Ensure there are no issues with NaNs in the group
df_model = df.dropna(subset=[
    'dividend',
    'retail_eligible_i',
    'CL4_yes',
    'Management_Group'
]).copy()

df_model['Management_Group_clean'] = df_model['Management_Group'].astype(str)

# Model with clustering
model = smf.ols(
    formula="dividend ~ retail_eligible_i + CL4_yes",
    data=df_model
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_model['Management_Group_clean']}
)

print(model.summary())

Table 3. Fund Characteristics and Dividend Policies.

In [ ]:
import statsmodels.formula.api as smf

df_model = df.dropna(subset=[
    'dividend',
    'retail_eligible_i',
    'CL4_yes',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income',
    'Management_Group'
]).copy()

df_model['Management_Group_clean'] = df_model['Management_Group'].astype(str)

model_controls = smf.ols(
    formula="""
    dividend ~ retail_eligible_i + CL4_yes
    + log_tna + fund_age_years
    + is_equity + is_fixed_income
    """,
    data=df_model
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_model['Management_Group_clean']}
)

print(model_controls.summary())

Table 4. Within-Management Analysis.

In [ ]:
model_fe = smf.ols(
    formula="""
    dividend ~ retail_eligible_i + CL4_yes
    + log_tna + fund_age_years
    + is_equity + is_fixed_income
    + C(Management_Group)
    """,
    data=df
).fit()

print(model_fe.summary())

Table 5. Retail Targeting, Asset Class Interaction, and Management Effects

Without FE

In [ ]:
df['Management_Group_clean'] = df['Management_Group'].fillna('Missing').astype(str)

In [ ]:
import statsmodels.formula.api as smf

# 1. Required variables ONLY
vars_t5 = [
    'dividend',
    'retail_eligible_i',
    'is_fixed_income',
    'Management_Group_clean'
]

# 2. ONE SAMPLE FOR EVERYTHING
df_t5 = df[vars_t5].dropna()

# 3. Model
model_t5 = smf.ols(
    formula="dividend ~ retail_eligible_i * is_fixed_income",
    data=df_t5
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_t5['Management_Group_clean']}
)

print(model_t5.summary())

With FE

In [ ]:
import statsmodels.formula.api as smf

# 1. Cleaned cluster variable
df['Management_Group_clean'] = df['Management_Group'].astype('category').cat.codes

# 2. Required variables
vars_t5 = [
    'dividend',
    'retail_eligible_i',
    'is_fixed_income',
    'Management_Group',
    'Management_Group_clean'
]

# 3. One size fits all
df_t5 = df[vars_t5].dropna()

# 4. Model with FE
model = smf.ols(
    formula="dividend ~ retail_eligible_i * is_fixed_income + C(Management_Group)",
    data=df_t5
).fit()

# 5. Aligned cluster
model_t5 = model.get_robustcov_results(
    cov_type='cluster',
    groups=df_t5.loc[model.model.data.row_labels, 'Management_Group_clean']
)

print(model_t5.summary())

Table 6. Retail Targeting vs Economic Accessibility.

Without FE

In [ ]:
vars_t6 = [
    'dividend',
    'retail_eligible_i',
    'log_min_inv',
    'Management_Group_clean'
]

df_t6 = df[vars_t6].dropna()

model = smf.ols(
    formula="dividend ~ retail_eligible_i + log_min_inv",
    data=df_t6
).fit()

model_t6 = model.get_robustcov_results(
    cov_type='cluster',
    groups=df_t6.loc[model.model.data.row_labels, 'Management_Group_clean']
)

print(model_t6.summary())

With FE

In [ ]:
vars_t6_fe = [
    'dividend',
    'retail_eligible_i',
    'log_min_inv',
    'Management_Group',
    'Management_Group_clean'
]

df_t6_fe = df[vars_t6_fe].dropna()

model = smf.ols(
    formula="dividend ~ retail_eligible_i + log_min_inv + C(Management_Group)",
    data=df_t6_fe
).fit()

model_t6_fe = model.get_robustcov_results(
    cov_type='cluster',
    groups=df_t6_fe.loc[model.model.data.row_labels, 'Management_Group_clean']
)

print(model_t6_fe.summary())

Table A.1 – Sample Comparison

In [ ]:
df_full = pd.read_excel("ucits_dividend_data.xlsx")

vars_model = [
    'dividend_paid',
    'retail_eligible_i',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income'
]

df_final = df_full[vars_model].dropna()

# -----------------------------
# SAMPLE COMPARISON
# -----------------------------
vars_compare = ['log_tna', 'fund_age_years', 'is_equity', 'is_fixed_income']

summary_full = df_full[vars_compare].mean()
summary_final = df_final[vars_compare].mean()

comparison = pd.DataFrame({
    'Full sample': summary_full,
    'Final sample': summary_final,
    'Difference': summary_final - summary_full
})

print(comparison)

Table A.2. Alternative Measure of Economic Accessibility

In [ ]:
# Subsample
vars_model = [
    'dividend_paid',
    'retail_eligible_i',
    'high_min_inv',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income',
    'Management_Group'
]

df_base = df[vars_model].dropna()

# Model
model_alt = smf.ols(
    "dividend_paid ~ retail_eligible_i + high_min_inv + log_tna + fund_age_years + is_equity + is_fixed_income",
    data=df_base
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_base['Management_Group']}
)

# 5. Output
print(model_alt.summary())

Table A.3. Dividend Intensity and Retail Targeting

In [ ]:
# Controls
controls = "log_tna + fund_age_years + is_equity + is_fixed_income"

vars_base = [
    'dividend_paid',
    'retail_eligible_i',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income',
    'Management_Group'
]

df_base = df[vars_base].dropna()

# -----------------------------
# (1) BASELINE MODEL
# -----------------------------
model1 = smf.ols(
    f"dividend_paid ~ retail_eligible_i + {controls}",
    data=df_base
).fit()

res1 = model1.get_robustcov_results(
    cov_type='cluster',
    groups=df_base.loc[model1.model.data.row_labels, 'Management_Group']
)

# -----------------------------
# (2) INTENSITY (PAYING MEMBERS ONLY)
# -----------------------------
df_paid = df_base.copy()
df_paid['log_div_value'] = df['log_div_value']

df_paid = df_paid[
    (df_paid['dividend_paid'] == 1) &
    (df_paid['log_div_value'].notnull())
]

model2 = smf.ols(
    f"log_div_value ~ retail_eligible_i + {controls}",
    data=df_paid
).fit()

res2 = model2.get_robustcov_results(
    cov_type='cluster',
    groups=df_paid.loc[model2.model.data.row_labels, 'Management_Group']
)

# Output
print("\n(1) Dividend (Baseline)")
print(res1.summary())

print("\n(2) Log Dividend Value (Paid Funds)")
print(res2.summary())